In [2]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset

# 파일 업로드
uploaded = files.upload()

# 데이터프레임으로 파일 읽기
df = pd.read_csv('/content/cleaned_labeling_test.csv')

Saving cleaned_labeling.csv to cleaned_labeling.csv


In [10]:
# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# Tokenizer 준비
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True)

# Huggingface Dataset 변환
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Data collator (자동 padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model 준비
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Training arguments 설정
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# Trainer 준비
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 학습 시작
trainer.train()

# 평가
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(val_labels, preds))

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-10-7e956167ab21>:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shjeong020208 (shjeong020208-dong-eui-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,0.670199
2,No log,0.674886
3,No log,0.630613


              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4



In [13]:
# Google Drive 연동하여 저장

from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained('/content/drive/MyDrive/pan12_bert_test_model')
tokenizer.save_pretrained('/content/drive/MyDrive/pan12_bert_test_model')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/pan12_bert_test_model/tokenizer_config.json',
 '/content/drive/MyDrive/pan12_bert_test_model/special_tokens_map.json',
 '/content/drive/MyDrive/pan12_bert_test_model/vocab.txt',
 '/content/drive/MyDrive/pan12_bert_test_model/added_tokens.json')

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# 저장된 모델 경로 지정
model_path = '/content/drive/MyDrive/pan12_bert_test_model'

# 모델 및 토크나이저 불러오기
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

# GPU 사용 가능하면 이동
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 예측 테스트
#text = "0: hey u on? [SEP] 1: Hey baby [SEP] 1: What u doin [SEP] 0: hey [SEP] 1: Everything ok? [SEP] 0: um i guess so [SEP] 1: U have a good day? [SEP] 0: it was ok [SEP] 1: I missed u baby [SEP] 0: i broke a dish and my gramz is pissed off [SEP] 0: i miss u to [SEP] 1: Im sorry baby what dish was it? [SEP] 0: some china dish thing she had 4 like a 1000 years [SEP] 1: Oh [SEP] 1: That bites bad [SEP] 0: ya :( [SEP] 1: Wish i could make it better baby [SEP] 0: me too [SEP] 0: i hate this [SEP] 1: Wish u could just move in with me baby [SEP] 0: aww [SEP] 0: i wish that could happen [SEP] 0: it seems like 4eva until my dad gets back [SEP] 1: I can take care of u baby [SEP] 0: :(( [SEP] 1: Wish your dad could live down here near me so we can be together [SEP] 0: ya [SEP] 0: but he got a job in nj [SEP] 0: and i hate it here [SEP] 0: sorry [SEP] 1: I hope i would be a reason u wanna stay [SEP] 0: ya [SEP] 0: i cant wait til we meet [SEP] 1: I wish u would be my girl [SEP] 0: that wud b sweet [SEP] 1: Me either baby [SEP] 1: If your gramz leave for the day would it be better if u sneak off with me or we stay there at your place? [SEP] 0: idk [SEP] 0: maybe leave [SEP] 0: idk [SEP] 1: Ok [SEP] 0: if we stayed what u wanna do? [SEP] 1: I wish u were here now baby [SEP] 1: We could watch movies maybe make out or stuff [SEP] 0: if we left weher wud we go? [SEP] 1: Go shoppin for stuff for u then lunch then maybe go somewhere to be alone [SEP] 0: cool [SEP] 1: What u wanna shop for baby? [SEP] 0: what u wanna buy me?:-P [SEP] 1: How about some lingerie baby [SEP] 0: huh [SEP] 1: Lingerie is sexy bras and panties and thongs [SEP] 0: oh ok lol [SEP] 0: ya thatd b sweet [SEP] 1: I want my baby lookin very sexy [SEP] 0: o ya [SEP] 0: ima <3 that [SEP] 1: I want u to be if u want [SEP] 0: ya i do [SEP] 0: i need some fun [SEP] 1: U could have fun with me baby [SEP] 0: ya i want to [SEP] 1: :-* [SEP] 1: What all u wearin baby? [SEP] 0: jeans and a hoodie [SEP] 1: What kind bra and panties baby? [SEP] 0: lol um lemme look [SEP] 1: I could warm u up if i was with u [SEP] 0: blue stripe panties [SEP] 0: and a white bra [SEP] 0: my feet are always cold [SEP] 1: If u were here i would want u only in your bra and panties baby [SEP] 0: lol [SEP] 0: could i wear sox?:-P [SEP] 1: Yes but u may not need them baby [SEP] 0: awww [SEP] 0: u gonna rub my feet so there warm? [SEP] 1: We would be under a quilt makin out hot and heavy [SEP] 0: i think my feet would be cold neway [SEP] 0: lol [SEP] 1: Ill keep u warm all over baby [SEP] 0: i hope so [SEP] 1: Ill rub my legs over them warmin them up while we kiss [SEP] 0: ok <3 [SEP] 1: And my hands will rub u in other places warmin u up baby [SEP] 0: lol like where [SEP] 1: Your back, sides, your butt maybe in your panties [SEP] 0: oh [SEP] 1: You like baby? [SEP] 0: ya [SEP] 1: I want u baby [SEP] 0: u do? [SEP] 1: Yes do u want me baby? [SEP] 0: ya i do [SEP] 0: im lonely [SEP] 0: and i need sum1 to treat me nice [SEP] 1: Baby ill treat u like a princess [SEP] 0: ya [SEP] 0: i wanna b one again [SEP] 1: I hope u could tell that when u have talked to me [SEP] 0: ya [SEP] 1: Baby i would treat u as my princess when im with u baby [SEP] 0: u would be my prince? [SEP] 1: Yes baby [SEP] 1: Do u want me to be your prince? [SEP] 0: YA [SEP] 0: dang my gramz here [SEP] 0: i got to go now [SEP] 0: will u be on tomorow? [SEP] 1: Love ya baby [SEP] 0: :-* [SEP] 1: Yes baby [SEP] 1: :-*"
text = "0: hey u on? 1: Hey baby 1: What u doin 0: hey 1: Everything ok? 0: um i guess so 1: U have a good day? 0: it was ok 1: I missed u baby 0: i broke a dish and my gramz is pissed off 0: i miss u to 1: Im sorry baby what dish was it? 0: some china dish thing she had 4 like a 1000 years 1: Oh 1: That bites bad 0: ya :( 1: Wish i could make it better baby 0: me too 0: i hate this 1: Wish u could just move in with me baby 0: aww 0: i wish that could happen 0: it seems like 4eva until my dad gets back 1: I can take care of u baby 0: :(( 1: Wish your dad could live down here near me so we can be together 0: ya 0: but he got a job in nj 0: and i hate it here 0: sorry 1: I hope i would be a reason u wanna stay 0: ya 0: i cant wait til we meet 1: I wish u would be my girl 0: that wud b sweet 1: Me either baby 1: If your gramz leave for the day would it be better if u sneak off with me or we stay there at your place? 0: idk 0: maybe leave 0: idk 1: Ok 0: if we stayed what u wanna do? 1: I wish u were here now baby 1: We could watch movies maybe make out or stuff 0: if we left weher wud we go? 1: Go shoppin for stuff for u then lunch then maybe go somewhere to be alone 0: cool 1: What u wanna shop for baby? 0: what u wanna buy me?:-P 1: How about some lingerie baby  0: huh  1: Lingerie is sexy bras and panties and thongs  0: oh ok lol  0: ya thatd b sweet  1: I want my baby lookin very sexy  0: o ya  0: ima <3 that  1: I want u to be if u want  0: ya i do  0: i need some fun  1: U could have fun with me baby  0: ya i want to  1: :-*  1: What all u wearin baby?  0: jeans and a hoodie  1: What kind bra and panties baby?  0: lol um lemme look  1: I could warm u up if i was with u  0: blue stripe panties  0: and a white bra  0: my feet are always cold  1: If u were here i would want u only in your bra and panties baby  0: lol  0: could i wear sox?:-P  1: Yes but u may not need them baby  0: awww  0: u gonna rub my feet so there warm?  1: We would be under a quilt makin out hot and heavy  0: i think my feet would be cold neway  0: lol  1: Ill keep u warm all over baby  0: i hope so  1: Ill rub my legs over them warmin them up while we kiss  0: ok <3  1: And my hands will rub u in other places warmin u up baby  0: lol like where  1: Your back, sides, your butt maybe in your panties 0: oh 1: You like baby? 0: ya 1: I want u baby 0: u do? 1: Yes do u want me baby? 0: ya i do 0: im lonely 0: and i need sum1 to treat me nice 1: Baby ill treat u like a princess 0: ya 0: i wanna b one again 1: I hope u could tell that when u have talked to me 0: ya 1: Baby i would treat u as my princess when im with u baby 0: u would be my prince? 1: Yes baby 1: Do u want me to be your prince? 0: YA 0: dang my gramz here 0: i got to go now 0: will u be on tomorow? 1: Love ya baby 0: :-* 1: Yes baby 1: :-*"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

outputs = model(**inputs)
logits = outputs.logits
predicted_class = logits.argmax().item()

print(f"Predicted class: {predicted_class} (0=non-grooming, 1=grooming)")
